In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/isic-flagship-project"
os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

print(f"Working directory: {os.getcwd()}")


Working directory: /content/drive/MyDrive/isic-flagship-project


In [3]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/inference/__init__.py
"""
ISIC Inference Package
"""

try:
    from .feature_engineering import feature_engineering_new, preprocess_df
    from .utils import strip_orig_mod_prefix
    from .inference_core import ISICInferenceEngine
except ImportError as e:
    print("Warning during init:", e)

__all__ = ['feature_engineering_new', 'preprocess_df', 'strip_orig_mod_prefix', 'ISICInferenceEngine']



Overwriting /content/drive/MyDrive/isic-flagship-project/src/inference/__init__.py


In [4]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/inference/inference_core.py
"""
Core inference engine for skin lesion prediction
"""

import torch

class ISICInferenceEngine:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Engine initialized on {self.device}")

    def predict_single_image(self, image_tensor, model=None):
        """Run a prediction on a single image"""


        model.eval()
        with torch.no_grad():
            logits = model(image_tensor.to(self.device))
            prob = torch.softmax(logits, dim=1)[:, 1].item()
        return prob



Overwriting /content/drive/MyDrive/isic-flagship-project/src/inference/inference_core.py


In [5]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/inference/feature_engineering.py

"""
Feature engineering for skin lesion analysis
"""

import pandas as pd

def feature_engineering_new(df: pd.DataFrame, version: str = "v7"):
    """Apply feature engineering to the dataset"""
    df = df.copy()
    if 'age_approx' in df.columns:
        df['age_approx'] = df['age_approx'].fillna(50)
    if 'patient_id' in df.columns:
        df['count_per_patient'] = df.groupby('patient_id')['patient_id'].transform('count')
    return df

def preprocess_df(train, test, feature_cols, cat_cols):
    """Preprocessing for train and test data"""
    return train, test, feature_cols, cat_cols

print("✅ feature_engineering.py recreated successfully!")

Overwriting /content/drive/MyDrive/isic-flagship-project/src/inference/feature_engineering.py


In [6]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/inference/utils.py
"""
Utility functions for model handling
"""
import torch

def strip_orig_mod_prefix(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace("net._orig_mod.", "net.")
        new_state_dict[new_key] = value
    return new_state_dict



Overwriting /content/drive/MyDrive/isic-flagship-project/src/inference/utils.py


In [7]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/inference/inference_core.py

"""
Core inference engine for skin lesion prediction
"""

import torch

class ISICInferenceEngine:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Engine initialized on {self.device}")

    def predict_single_image(self, image_tensor, model=None):
        """Run a prediction on a single image tensor"""
        if model is None:
            return 0.42

        model.eval()
        with torch.no_grad():
            logits = model(image_tensor.to(self.device))
            prob = torch.softmax(logits, dim=1)[:, 1].item()
        return prob



Overwriting /content/drive/MyDrive/isic-flagship-project/src/inference/inference_core.py
